# Comparative Epidemiological Study of COVID-19 Waves & Vaccine Portfolios (EU/EEA, 2020–2022)

**Author:** Bilal Abdul (<abac23@student.bth.se>)  
**Institution:** Blekinge Institute of Technology (BTH), Sweden  
**Project Type:** University Data Science Research Project  

---

### Project Abstract
This research notebook presents a comprehensive statistical study and exploratory data analysis (EDA) of the COVID-19 pandemic across 30 EU/EEA countries between January 2020 and December 2022. Using epidemiological records from the European Centre for Disease Prevention and Control (ECDC), this project builds data engineering pipelines and statistical models to investigate infection waves, vaccine brand allocations, clinical capacity burdens, and vaccine skepticism dynamics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import pearsonr, wilcoxon

# Set up custom plot styling
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Liberation Sans']
plt.rcParams['axes.edgecolor'] = '#BDC3C7'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['xtick.color'] = '#2C3E50'
plt.rcParams['ytick.color'] = '#2C3E50'
plt.rcParams['grid.color'] = '#ECF0F1'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.alpha'] = 0.5

# Configuration
base_path = "data"
plots_dir = "plots"
os.makedirs(plots_dir, exist_ok=True)

cases_file = os.path.join(base_path, "1.COVID-19_daily_number_of_new_cases_and_deaths.csv")
vacc_file = os.path.join(base_path, "2.COVID-19_vaccination.csv")
hosp_file = os.path.join(base_path, "3.COVID-19_hospital_and_ICU_admission_rates.csv")

print("Ingesting and clean-mapping ECDC datasets...")
df_cases = pd.read_csv(cases_file, encoding='utf-8-sig')
df_vacc = pd.read_csv(vacc_file, encoding='utf-8-sig')
df_hosp = pd.read_csv(hosp_file, encoding='utf-8-sig')

# Preprocessing:
# Standardize cases dateRep
df_cases['date'] = pd.to_datetime(df_cases['dateRep'], format='%d/%m/%Y')
df_cases = df_cases[(df_cases['date'].dt.year >= 2020) & (df_cases['date'].dt.year <= 2022)]

# Deduplicate regional vaccination rows
# NUTS-2 subnational lines duplicate national totals. We filter where Region == ReportingCountry.
df_vacc_nat = df_vacc[df_vacc['Region'] == df_vacc['ReportingCountry']].copy()

# Standardize country names to geoId
country_map = df_cases[['countriesAndTerritories', 'geoId']].drop_duplicates().set_index('countriesAndTerritories')['geoId'].to_dict()
country_map.update({
    'Czech Republic': 'CZ',
    'Czechia': 'CZ',
    'Greece': 'EL',
    'Slovakia': 'SK'
})

df_hosp['geoId'] = df_hosp['country'].map(country_map)
df_hosp = df_hosp[df_hosp['geoId'].notna()]

# Coordinate mapping for visual bubble projections
country_coords = {
    'AT': (47.5162, 14.5501), 'BE': (50.5039, 4.4699), 'BG': (42.7339, 25.4858),
    'CY': (35.1264, 33.4299), 'CZ': (49.8175, 15.4730), 'DE': (51.1657, 10.4515),
    'DK': (56.2639, 9.5018),  'EE': (58.5953, 25.0136), 'EL': (39.0742, 21.8243),
    'ES': (40.4637, -3.7492), 'FI': (61.9241, 25.7482), 'FR': (46.2276, 2.2137),
    'HR': (45.1000, 15.2000), 'HU': (47.1625, 19.5033), 'IE': (53.4129, -8.2439),
    'IS': (64.9631, -19.0208), 'IT': (41.8719, 12.5674), 'LI': (47.1660, 9.5554),
    'LT': (55.1694, 23.8813), 'LU': (49.8153, 6.1296),  'LV': (56.8796, 24.6032),
    'MT': (35.9375, 14.3754), 'NL': (52.1326, 5.2913),  'NO': (60.4720, 8.4689),
    'PL': (51.9194, 19.1451), 'PT': (39.3999, -8.2245), 'RO': (45.9432, 24.9668),
    'SE': (60.1282, 18.6435), 'SI': (46.1512, 14.9955), 'SK': (48.6690, 19.6990)
}

vaccine_names = {
    'COM': 'Comirnaty (Pfizer/BioNTech)',
    'COMBA.1': 'Comirnaty Bivalent (Pfizer/BioNTech)',
    'COMBA.4-5': 'Comirnaty Bivalent (Pfizer/BioNTech)',
    'COMBIV': 'Comirnaty Bivalent (Pfizer/BioNTech)',
    'MOD': 'Spikevax (Moderna)',
    'MODBA.1': 'Spikevax Bivalent (Moderna)',
    'MODBA.4-5': 'Spikevax Bivalent (Moderna)',
    'MODBIV': 'Spikevax Bivalent (Moderna)',
    'AZ': 'Vaxzevria (AstraZeneca)',
    'JANSS': 'Jcovden (Janssen/J&J)',
    'NVXD': 'Nuvaxovid (Novavax)',
    'VLA': 'Valneva',
    'SPU': 'Sputnik V',
    'SIN': 'Sinopharm',
    'BECNBG': 'BBIBP-CorV (Sinopharm)',
    'BHACOV': 'Covaxin (Bharat Biotech)',
    'SGSK': 'Sanofi-GSK',
    'UNK': 'Unknown Vaccine'
}
print("Data loaded. Ingestion complete!")

## Section 3.1: Mandatory Questions

### Question 1: Top 10 Countries and Quarterly Waves
We find the 10 countries with the highest cumulative cases between 2020 and 2022, calculate what percentage of their population was infected, and plot the trends quarterly.

In [ ]:
# 1. Identify top 10 countries overall by total cases
country_totals = df_cases.groupby('countriesAndTerritories')['cases'].sum()
top_10_countries = country_totals.nlargest(10).index.tolist()

df_cases['year_quarter'] = df_cases['date'].dt.to_period('Q').astype(str)
df_top_10 = df_cases[df_cases['countriesAndTerritories'].isin(top_10_countries)]

q_cases = df_top_10.pivot_table(index='countriesAndTerritories', columns='year_quarter', values='cases', aggfunc='sum')
q_cases.to_csv(os.path.join(plots_dir, "q1_quarterly_cases.csv"))

# Population infection rate calculation
pop_df = df_cases[df_cases['countriesAndTerritories'].isin(top_10_countries)][['countriesAndTerritories', 'popData2020']].drop_duplicates().set_index('countriesAndTerritories')
total_cases = country_totals.loc[top_10_countries]

pop_compare = pd.DataFrame({
    'Total_Cases': total_cases,
    'Population_2020': pop_df['popData2020'],
    'Infection_Rate_Pct': (total_cases / pop_df['popData2020']) * 100
}).sort_values(by='Infection_Rate_Pct', ascending=False)

print("Infection Rate Relative to Total Population (Top 10):")
print(pop_compare.to_string())

# Visualise quarterly trends
fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#2C3E50', '#16A085', '#2980B9', '#8E44AD', '#D35400', '#C0392B', '#7F8C8D', '#1abc9c', '#3498db', '#9b59b6']

for i, country in enumerate(top_10_countries):
    series = q_cases.loc[country]
    ax.plot(series.index, series.values / 1e6, label=country, color=colors[i], marker='o', linewidth=2.0, markersize=5)
    
ax.set_title("Quarterly COVID-19 Case Wave Trajectories (Top 10 Countries)", fontsize=12, fontweight='bold', color='#2C3E50', pad=15)
ax.set_xlabel("Year & Quarter", fontsize=10)
ax.set_ylabel("New Cases (Millions)", fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDC3C7')
ax.spines['bottom'].set_color('#BDC3C7')
ax.yaxis.grid(True)
ax.legend(frameon=False, loc='upper left', ncol=2, fontsize=9.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Question 2: Spatial Bubble Map
We plot the cumulative cases and deaths on a custom coordinate-based bubble map representing Europe to see the geographic spread in 2020, 2021, and 2022.

In [ ]:
df_cases['year_str'] = df_cases['date'].dt.year.astype(str)
df_cases['geoId'] = df_cases['countriesAndTerritories'].map(country_map)
df_cases_filtered = df_cases[df_cases['geoId'].notna()].copy()

# Select 2021 to display as an illustrative spatio-temporal map
year = '2021'
df_year = df_cases_filtered[df_cases_filtered['year_str'] == year]
geo_agg = df_year.groupby('geoId').agg({'cases': 'sum', 'deaths': 'sum'}).reset_index()
geo_agg['lat'] = geo_agg['geoId'].map(lambda x: country_coords[x][0] if x in country_coords else np.nan)
geo_agg['lon'] = geo_agg['geoId'].map(lambda x: country_coords[x][1] if x in country_coords else np.nan)
geo_agg = geo_agg.dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5), facecolor='white')

# Cases bubble map
ax = axes[0]
ax.set_facecolor('#F8F9FA')
sc1 = ax.scatter(geo_agg['lon'], geo_agg['lat'], s=geo_agg['cases'] / 1e4 * 1.5 + 40,
                 c=geo_agg['cases'] / 1e6, cmap='Blues', alpha=0.8, edgecolors='#34495E', linewidths=0.8, zorder=2)
ax.set_title(f"Cumulative Cases ({year})", fontsize=11, fontweight='bold', color='#2C3E50')
for idx, row in geo_agg.iterrows():
    ax.annotate(row['geoId'], (row['lon'], row['lat']), fontsize=7.5, color='#2C3E50', weight='bold', ha='center', va='center')
ax.grid(True)
ax.set_xlim(-25, 35)
ax.set_ylim(34, 72)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
cbar1 = fig.colorbar(sc1, ax=ax, orientation='horizontal', pad=0.08, shrink=0.7)
cbar1.set_label("Cases (Millions)")

# Deaths bubble map
ax = axes[1]
ax.set_facecolor('#F8F9FA')
sc2 = ax.scatter(geo_agg['lon'], geo_agg['lat'], s=geo_agg['deaths'] / 1e2 * 1.5 + 40,
                 c=geo_agg['deaths'] / 1e3, cmap='Reds', alpha=0.8, edgecolors='#34495E', linewidths=0.8, zorder=2)
ax.set_title(f"Cumulative Deaths ({year})", fontsize=11, fontweight='bold', color='#2C3E50')
for idx, row in geo_agg.iterrows():
    ax.annotate(row['geoId'], (row['lon'], row['lat']), fontsize=7.5, color='#2C3E50', weight='bold', ha='center', va='center')
ax.grid(True)
ax.set_xlim(-25, 35)
ax.set_ylim(34, 72)
ax.set_xlabel("Longitude (°E)")
cbar2 = fig.colorbar(sc2, ax=ax, orientation='horizontal', pad=0.08, shrink=0.7)
cbar2.set_label("Deaths (Thousands)")

plt.suptitle(f"Spatio-Temporal Distribution of COVID-19 in the EU/EEA ({year})", fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### Question 3: Top 3 Vaccine Brands and Exceptions
We find the top 3 most popular vaccine brands across the EU/EEA by doses administered and analyze if there are any countries with different popularity trends.

In [ ]:
dose_cols = [c for c in df_vacc_nat.columns if 'Dose' in c and c != 'FirstDoseRefused' and c != 'UnknownDose']
df_vacc_nat['Total_Doses_Administered'] = df_vacc_nat[dose_cols].sum(axis=1)

brand_agg = df_vacc_nat[df_vacc_nat['TargetGroup'] == 'ALL'].groupby('Vaccine')['Total_Doses_Administered'].sum()
brand_shares = (brand_agg / brand_agg.sum()) * 100
top_brands = brand_agg.nlargest(5)

print("Top COVID-19 Vaccine Brands in the EU/EEA:")
for vaccine, doses in top_brands.items():
    name = vaccine_names.get(vaccine, vaccine)
    share = brand_shares.loc[vaccine]
    print(f"- {name} ({vaccine}): {doses:,.0f} doses ({share:.2f}%)")

top_3_overall = brand_agg.nlargest(3).index.tolist()

country_brand_agg = df_vacc_nat[df_vacc_nat['TargetGroup'] == 'ALL'].groupby(['ReportingCountry', 'Vaccine'])['Total_Doses_Administered'].sum().unstack(fill_value=0)
print("\nCountries with Brand Popularity Exceptions:")
for country in country_brand_agg.index:
    country_doses = country_brand_agg.loc[country]
    top_3_country = country_doses.nlargest(3).index.tolist()
    if not all(b in top_3_overall for b in top_3_country):
        country_shares = (country_doses / country_doses.sum()) * 100
        print(f"* Country: {country}")
        for b in top_3_country:
            print(f"  - {vaccine_names.get(b, b)}: {country_shares[b]:.1f}%")

### Question 4: Target Groups for the Top 3 Brands
We analyze the distribution of doses for the top 3 vaccine brands across age and occupational target groups.

In [ ]:
agg_groups = ['ALL', '1_Age60+', '1_Age<60', 'AgeUNK']
specific_vacc_df = df_vacc_nat[~df_vacc_nat['TargetGroup'].isin(agg_groups)].copy()
tg_vacc_agg = specific_vacc_df.groupby(['Vaccine', 'TargetGroup'])['Total_Doses_Administered'].sum().unstack(fill_value=0)

fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True)
colors = ['#2C3E50', '#26A69A', '#E74C3C']

for i, vaccine in enumerate(top_3_overall):
    ax = axes[i]
    shares = (tg_vacc_agg.loc[vaccine] / tg_vacc_agg.loc[vaccine].sum()) * 100
    shares.plot(kind='bar', ax=ax, color=colors[i], edgecolor='white', alpha=0.9)
    ax.set_title(f"Demographic Distribution for {vaccine_names.get(vaccine, vaccine)}", fontsize=11, fontweight='bold')
    ax.set_ylabel("Share of Doses (%)")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True)

plt.xlabel("Demographic Target Groups")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Question 5: Vaccine Skepticism vs. Peak Hospitalization
We define a vaccine skepticism rate (1 - first dose coverage) and perform a Pearson correlation test against peak weekly hospital admissions per 100k in 2021.

In [ ]:
results = []
for country in df_vacc_nat['ReportingCountry'].unique():
    sub = df_vacc_nat[df_vacc_nat['ReportingCountry'] == country]
    pop = sub['Population'].max()
    if pop == 0 or np.isnan(pop): continue
    
    adult_first = sub[sub['TargetGroup'] == 'ALL']['FirstDose'].sum()
    child_first = sub[sub['TargetGroup'] == 'Age<18']['FirstDose'].sum() if 'Age<18' in sub['TargetGroup'].unique() else \
                  sub[sub['TargetGroup'].isin(['Age15_17', 'Age10_14', 'Age5_9', 'Age0_4'])]['FirstDose'].sum()
                  
    total_first = adult_first + child_first
    first_rate = total_first / pop
    results.append({
        'geoId': country,
        'Population': pop,
        'Skepticism_Rate': 1.0 - first_rate
    })
df_skep = pd.DataFrame(results)

df_hosp_filtered = df_hosp[df_hosp['indicator'] == 'Weekly new hospital admissions per 100k'].copy()
df_hosp_filtered['date_dt'] = pd.to_datetime(df_hosp_filtered['date'])
df_hosp_2021 = df_hosp_filtered[df_hosp_filtered['date_dt'].dt.year == 2021]
hosp_burden = df_hosp_2021.groupby('geoId')['value'].max().reset_index().rename(columns={'value': 'Peak_Weekly_Hosp_per_100k_2021'})

merged = pd.merge(df_skep, hosp_burden, on='geoId')

# Pearson correlation
r_coeff, p_value = pearsonr(merged['Skepticism_Rate'], merged['Peak_Weekly_Hosp_per_100k_2021'])
print(f"Pearson Correlation Coefficient: r = {r_coeff:.4f} (p-value = {p_value:.4f})")

# Plotting
fig, ax = plt.subplots(figsize=(8.5, 5.5))
sns.regplot(data=merged, x='Skepticism_Rate', y='Peak_Weekly_Hosp_per_100k_2021',
            scatter_kws={'s': merged['Population'] / 5e5 + 40, 'color': '#2C3E50', 'alpha': 0.7, 'edgecolors': '#34495E'},
            line_kws={'color': '#E74C3C', 'linewidth': 1.5, 'label': f'Fit (r={r_coeff:.2f})'}, ax=ax)

for idx, row in merged.iterrows():
    ax.annotate(row['geoId'], (row['Skepticism_Rate'], row['Peak_Weekly_Hosp_per_100k_2021']),
                textcoords="offset points", xytext=(0, 5), ha='center', fontsize=8, weight='bold')

ax.set_title("Vaccine Skepticism vs. Peak Weekly Hospitalizations (2021)", fontsize=11, fontweight='bold')
ax.set_xlabel("Vaccine Skepticism Rate (1 - First Dose Coverage)")
ax.set_ylabel("Peak Weekly Hospital Admissions per 100k (2021)")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Question 6: Under-18 First Dose Vaccination Ranks
We rank the countries by the proportion of vaccinated people under age 18 relative to their total populations.

In [ ]:
results = []
for country in df_vacc_nat['ReportingCountry'].unique():
    sub = df_vacc_nat[df_vacc_nat['ReportingCountry'] == country]
    pop = sub['Population'].max()
    if pop == 0 or np.isnan(pop): continue
    
    child_first = sub[sub['TargetGroup'] == 'Age<18']['FirstDose'].sum() if 'Age<18' in sub['TargetGroup'].unique() else \
                  sub[sub['TargetGroup'].isin(['Age15_17', 'Age10_14', 'Age5_9', 'Age0_4'])]['FirstDose'].sum()
                  
    results.append({
        'geoId': country,
        'Under18_Rate_of_Total_Pop': child_first / pop
    })
df_u18 = pd.DataFrame(results).sort_values(by='Under18_Rate_of_Total_Pop', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5.5))
colors = ['#2E7D32' if i < 3 else ('#C62828' if i >= len(df_u18)-3 else '#7F8C8D') for i in range(len(df_u18))]
bars = ax.bar(df_u18['geoId'], df_u18['Under18_Rate_of_Total_Pop'] * 100, color=colors, edgecolor='white', alpha=0.9)

ax.set_title("Under-18 First-Dose Vaccination Rate Relative to Country Population", fontsize=11, fontweight='bold')
ax.set_ylabel("Vaccinated Under-18 Share (%)")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### Question 7: Weighted Average Age of Second-Dose Recipients
We calculate the weighted average age of second-dose recipients for each country, including a fallback for Germany because it only reports coarse target groups.

In [ ]:
age_midpoints = {
    'Age0_4': 2.5, 'Age5_9': 7.0, 'Age10_14': 12.0, 'Age15_17': 16.0,
    'Age<18': 9.0, 'Age18_24': 21.0, 'Age25_49': 37.0, 'Age50_59': 54.5,
    'Age60_69': 64.5, 'Age70_79': 74.5, 'Age80+': 85.0
}

non_age_groups = ['ALL', '1_Age60+', '1_Age<60', 'HCW', 'LTCF', 'AgeUNK']
age_vacc = df_vacc_nat[~df_vacc_nat['TargetGroup'].isin(non_age_groups)].copy()
age_vacc = age_vacc[age_vacc['TargetGroup'].isin(age_midpoints.keys())]
age_vacc['Midpoint'] = age_vacc['TargetGroup'].map(age_midpoints)

country_ages = []
for country in df_vacc_nat['ReportingCountry'].unique():
    country_data = df_vacc_nat[df_vacc_nat['ReportingCountry'] == country]
    unique_groups = country_data['TargetGroup'].unique()
    has_granular = any(g in unique_groups for g in ['Age18_24', 'Age25_49', 'Age50_59', 'Age80+'])
    
    if has_granular:
        sub = age_vacc[age_vacc['ReportingCountry'] == country]
        weighted_sum = (sub['SecondDose'] * sub['Midpoint']).sum()
        total_second = sub['SecondDose'].sum()
    else:
        # Coarse fallback for Germany (DE)
        coarse_midpoints = {'Age<18': 9.0, '1_Age<60': 39.0, '1_Age60+': 72.5}
        sub = country_data[country_data['TargetGroup'].isin(coarse_midpoints.keys())].copy()
        sub['Midpoint'] = sub['TargetGroup'].map(coarse_midpoints)
        weighted_sum = (sub['SecondDose'] * sub['Midpoint']).sum()
        total_second = sub['SecondDose'].sum()
        
    if total_second == 0: continue
    
    country_ages.append({
        'geoId': country,
        'Weighted_Average_Age': weighted_sum / total_second
    })
df_ages = pd.DataFrame(country_ages).sort_values(by='Weighted_Average_Age', ascending=False)

print("Oldest Second-Dose Vaccinated Populations (Top 5):")
print(df_ages.head(5).to_string(index=False))
print("\nYoungest Second-Dose Vaccinated Populations (Top 5):")
print(df_ages.tail(5).to_string(index=False))

### Question 8: Clinical Capacity and Healthcare Burden (2020 vs. 2022)
We compare the peak daily hospital and ICU occupancy rates per 100k population between 2020 (pre-vaccine) and 2022 (post-vaccine).

In [ ]:
df_hosp['date_dt'] = pd.to_datetime(df_hosp['date'])
df_hosp['year_str'] = df_hosp['date_dt'].dt.year.astype(str)

pop_mapping = df_cases[['geoId', 'popData2020']].drop_duplicates().set_index('geoId')['popData2020'].to_dict()
hosp_occ = df_hosp[df_hosp['indicator'].isin(['Daily hospital occupancy', 'Daily ICU occupancy'])].copy()
hosp_occ['Population'] = hosp_occ['geoId'].map(pop_mapping)
hosp_occ = hosp_occ.dropna(subset=['Population'])
hosp_occ['value_per_100k'] = (hosp_occ['value'] / hosp_occ['Population']) * 1e5

occ_agg = hosp_occ.groupby(['geoId', 'indicator', 'year_str'])['value_per_100k'].max().unstack(fill_value=np.nan)
occ_agg = occ_agg[['2020', '2022']].dropna().reset_index()

# Plot Daily ICU Occupancy Peak
sub_icu = occ_agg[occ_agg['indicator'] == 'Daily ICU occupancy'].sort_values(by='2020', ascending=False)
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(sub_icu))
width = 0.35

ax.bar(x - width/2, sub_icu['2020'], width, label='2020 (Pre-vaccine)', color='#34495E', alpha=0.9)
ax.bar(x + width/2, sub_icu['2022'], width, label='2022 (Omicron)', color='#26A69A', alpha=0.9)

ax.set_title("Peak Daily ICU Occupancy per 100k (2020 vs 2022)", fontsize=11, fontweight='bold')
ax.set_ylabel("Occupancy per 100k")
ax.set_xticks(x)
ax.set_xticklabels(sub_icu['geoId'], fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Section 3.2: Advanced Questions (Grades A & B)

### Advanced Question A: Time-Lagged Correlation between Vaccination and CFR in Germany
We calculate the rolling weekly Case Fatality Rate (CFR) in Germany and run a Pearson correlation test against cumulative second-dose vaccination rate at different lag offsets (0 to 8 weeks) to model the delay in vaccine protection.

In [ ]:
de_cases = df_cases[df_cases['geoId'] == 'DE'].sort_values(by='date').copy()
de_vacc = df_vacc_nat[(df_vacc_nat['ReportingCountry'] == 'DE') & (df_vacc_nat['TargetGroup'] == 'ALL')].copy()

de_cases['year_week_iso'] = de_cases['date'].dt.strftime('%G-W%V')
cases_weekly = de_cases.groupby('year_week_iso').agg({'cases': 'sum', 'deaths': 'sum'}).reset_index()
vacc_weekly = de_vacc.groupby('YearWeekISO').agg({'FirstDose': 'sum', 'SecondDose': 'sum'}).reset_index()

weekly = pd.merge(cases_weekly, vacc_weekly, left_on='year_week_iso', right_on='YearWeekISO')
weekly = weekly.sort_values(by='year_week_iso').reset_index(drop=True)

de_pop = 83237124
weekly['SecondDose_Rate'] = weekly['SecondDose'].cumsum() / de_pop
weekly['rolling_cases'] = weekly['cases'].rolling(4, min_periods=1).sum()
weekly['rolling_deaths'] = weekly['deaths'].rolling(4, min_periods=1).sum()
weekly['CFR'] = (weekly['rolling_deaths'] / weekly['rolling_cases']) * 100
weekly['CFR'] = weekly['CFR'].fillna(0)

print("Lagged Pearson Correlation Coefficients (Germany):")
for lag in [0, 1, 2, 3, 4, 6, 8]:
    shifted_cfr = weekly['CFR'].shift(-lag)
    mask = weekly['SecondDose_Rate'].notna() & shifted_cfr.notna()
    r, p = pearsonr(weekly.loc[mask, 'SecondDose_Rate'], shifted_cfr[mask])
    print(f"  - Lag {lag} Weeks: r = {r:.4f} (p = {p:.4f})")

### Advanced Question B: Paired Wilcoxon Signed-Rank Test on Clinical Severity
We calculate the daily ICU occupancy as a share of hospital occupancy (clinical severity ratio) for each country and use a paired Wilcoxon Signed-Rank test to verify if there is a significant difference between 2020 and 2022.

In [ ]:
df_icu = df_hosp[df_hosp['indicator'] == 'Daily ICU occupancy'][['geoId', 'date', 'value']].rename(columns={'value': 'icu'})
df_h = df_hosp[df_hosp['indicator'] == 'Daily hospital occupancy'][['geoId', 'date', 'value']].rename(columns={'value': 'hosp'})

df_sev = pd.merge(df_icu, df_h, on=['geoId', 'date'])
df_sev['date_dt'] = pd.to_datetime(df_sev['date'])
df_sev['year_str'] = df_sev['date_dt'].dt.year.astype(str)

df_sev_filtered = df_sev[(df_sev['year_str'].isin(['2020', '2022'])) & (df_sev['hosp'] > 0)].copy()
df_sev_filtered['severity_ratio'] = df_sev_filtered['icu'] / df_sev_filtered['hosp']

country_sev = df_sev_filtered.groupby(['geoId', 'year_str'])['severity_ratio'].mean().unstack(fill_value=np.nan).dropna()

# Wilcoxon Signed-Rank Test
stat, p_val = wilcoxon(country_sev['2020'], country_sev['2022'])
print(f"Wilcoxon Signed-Rank Test Statistic: W = {stat:.1f} (p-value = {p_val:.5f})")
if p_val < 0.05:
    print("Conclusion: Reject the null hypothesis; clinical severity ratios decreased significantly from 2020 to 2022.")
else:
    print("Conclusion: Fail to reject the null hypothesis.")

### Advanced Question C: Unsupervised PCA and K-Means Clustering on Vaccine Portfolios
We normalize the vaccine brand shares for each country and use PCA for dimensionality reduction and K-Means clustering (K=3) to analyze how similar their vaccine procurement portfolios are.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

country_brand_agg = df_vacc_nat[df_vacc_nat['TargetGroup'] == 'ALL'].groupby(['ReportingCountry', 'Vaccine'])['Total_Doses_Administered'].sum().unstack(fill_value=0)
country_brand_shares = country_brand_agg.div(country_brand_agg.sum(axis=1), axis=0)

# PCA and K-Means
pca = PCA(n_components=2)
pca_res = pca.fit_transform(country_brand_shares)

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(country_brand_shares)

df_cluster = pd.DataFrame({
    'geoId': country_brand_shares.index,
    'PC1': pca_res[:, 0],
    'PC2': pca_res[:, 1],
    'Cluster': clusters
})

loadings = pca.components_
features = country_brand_shares.columns

country_brand_shares_reset = country_brand_shares.reset_index().rename(columns={'ReportingCountry': 'geoId'})
df_cluster = pd.merge(df_cluster, country_brand_shares_reset, on='geoId')

print("PCA Component Loadings (Vaccines driving PC1 and PC2):")
for f, pc1, pc2 in zip(features, loadings[0], loadings[1]):
    if abs(pc1) > 0.1 or abs(pc2) > 0.1:
        print(f"  - {vaccine_names.get(f, f)} ({f}): PC1={pc1:.3f}, PC2={pc2:.3f}")

print("\nGeopolitical Clusters:")
for c_id in range(3):
    sub = df_cluster[df_cluster['Cluster'] == c_id]
    print(f"  * Cluster {c_id+1} ({len(sub)} countries): {sub['geoId'].tolist()}")
    print("    Dominant vaccines:")
    mean_comp = sub[features].mean()
    for v, share in mean_comp.nlargest(2).items():
        print(f"      - {vaccine_names.get(v, v)}: {share*100:.1f}%")

## Summary & Key Takeaways

1. **Reduction in Clinical Severity**: The paired Wilcoxon test showed a highly significant drop in the ratio of ICU to hospital occupancy between 2020 and 2022 ($p = 0.0068$). This indicates that hospitalized patients required intensive care at a much lower rate during the Omicron wave compared to the pre-vaccine 2020 wave.
2. **Impact of Vaccination on Mortality**: In Germany, rolling weekly CFR had a very strong negative correlation with second-dose vaccination coverage ($r = -0.8236$, $p < 0.0001$), illustrating the real-world impact of the vaccination rollout.
3. **Vaccine Procurement Clustering**: PCA and K-Means clustering separated the 30 countries into three clear groups based on brand composition: a standardized EU portfolio (28 countries dominated by Pfizer), a Moderna outlier (Liechtenstein), and a database exception (Germany, which reported 100% of its doses as 'Unknown').
4. **Data Challenges**: Several data quality issues were identified, including Germany's lack of granular age data and use of coarse categories, subnational NUTS-2 double-counting risks in the vaccination dataset, and variations in national date and testing reporting.